In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tkinter as tk
import tkinter as ttk
from tkinter import Tk, Label, Entry, Button, Toplevel,simpledialog, messagebox
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.naive_bayes import GaussianNB
from sklearn.cluster import KMeans
from tkinter import messagebox
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import seaborn as sns
from mpl_toolkits.mplot3d import Axes3D
import geopandas as gpd
from sklearn.linear_model import LinearRegression
from tkinter.ttk import Style
from PIL import ImageTk, Image

data = pd.read_csv('dataset.csv')
df = pd.read_csv('dataset.csv')
columns_of_interest = ['country', 'year', 'electricity_generation']
electricity_generation_data = data[columns_of_interest]
electricity_generation_data = electricity_generation_data.dropna()
models = {}
for country in electricity_generation_data['country'].unique():
    country_data = electricity_generation_data[electricity_generation_data['country'] == country]
    X = country_data[['year']]
    y = country_data['electricity_generation']
    model = LinearRegression()
    model.fit(X, y)
    models[country] = model

def visualize_prediction(future_country, future_year, predicted_electricity_generation):

    graph_window = tk.Toplevel()
    graph_window.title("Electricity Generation Prediction")
    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot(111)

    country_data = electricity_generation_data[electricity_generation_data['country'] == future_country]
    ax.scatter(country_data['year'], country_data['electricity_generation'], color='blue', label='Actual')
    ax.scatter(future_year, predicted_electricity_generation, color='red', label='Predicted')
    ax.set_title(f'{future_country} - Year vs Electricity Generation')
    ax.set_xlabel('Year')
    ax.set_ylabel('Electricity Generation')
    ax.legend()
    ax.grid(True)

    canvas = FigureCanvasTkAgg(fig, master=graph_window)
    canvas.draw()
    canvas.get_tk_widget().pack()

def predict_electricity_generation():

    future_year = simpledialog.askinteger("Future Year", "Enter the future year:")
    future_country = simpledialog.askstring("Future Country", "Enter the future country:")
    scaling_factor = 2.5 
    if future_country in models:
        future_year_data = [[future_year]]
        predicted_electricity_generation = models[future_country].predict(future_year_data) * scaling_factor

        visualize_prediction(future_country, future_year, predicted_electricity_generation)
    else:
        print(f"No data available for {future_country}. Unable to make predictions.")
def visualize_country_activity(country_name, year):
    df = pd.read_csv('dataset.csv')
    country_year_df = df[(df['country'] == country_name) & (df['year'] == year)]
    if country_year_df.empty:
        messagebox.showerror("Error", f"No data available for {country_name} in the year {year}")
        return
    message = f"Energy-related data for {country_name} in {year}:\n\n"
    message += f"Electricity Generation: {country_year_df['electricity_generation'].values[0]}\n"
    message += f"Electricity Demand: {country_year_df['electricity_demand'].values[0]}\n"
    renewable_resources = country_year_df[['solar_electricity', 'wind_electricity', 'hydro_electricity', 'nuclear_electricity']].values[0]
    message += f"\nRenewable Energy Resources:\n{renewable_resources}\n"
    messagebox.showinfo("Summary", message)
    fig, axes = plt.subplots(2, 1, figsize=(10, 6))
    axes[0].bar(['Generation', 'Demand'], [country_year_df['electricity_generation'].values[0], country_year_df['electricity_demand'].values[0]], color=['blue', 'red'])
    axes[0].set_title(f'Electricity Generation vs Demand in {year}')
    axes[0].set_ylabel('Electricity (in TWh)')
    labels = ['Solar', 'Wind', 'Hydro', 'Nuclear']
    axes[1].pie(renewable_resources, labels=labels, autopct='%1.1f%%', startangle=90)
    axes[1].set_title(f'Renewable Energy Resources in {year}', pad=20)
    plt.tight_layout()
    plot_window = tk.Toplevel()
    plot_window.title(f'Energy Analysis for {country_name} in {year}')
    canvas = FigureCanvasTkAgg(fig, master=plot_window)
    canvas.draw()
    canvas.get_tk_widget().pack(side=tk.TOP, fill=tk.BOTH, expand=1)

def open_input_window():
    country = simpledialog.askstring("Country", "Enter Country:")
    year = simpledialog.askinteger("Year", "Enter Year:")
    if country and year:
        visualize_country_activity(country, year)

def open_input_window_top_countries():
    year = simpledialog.askinteger("Year", "Enter Year:")
    if year:
        top_countries_window = tk.Toplevel(root)
        top_countries_window.title(f"Top 5 Countries by Electricity Production in {year}")
        top_countries_window.geometry("800x600")
        top_countries_window.configure(bg="white")
        visualize_top_countries(year, top_countries_window)

def visualize_top_countries(year, top_countries_window):
    top_countries = top_countries_for_year(year)
    fig, ax = plt.subplots(figsize=(10, 6))
    top_countries.plot(kind='bar', color='skyblue', ax=ax)
    ax.set_title(f'Top 5 Countries by Electricity Production in {year}')
    ax.set_xlabel('Country')
    ax.set_ylabel('Electricity Production (TWh)')
    ax.grid(axis='y')
    fig.tight_layout()
    canvas = FigureCanvasTkAgg(fig, master=top_countries_window)
    canvas.draw()
    canvas.get_tk_widget().pack(side=tk.TOP, fill=tk.BOTH, expand=1)

def top_countries_for_year(year):
    year_df = electricity_generation_data[electricity_generation_data['year'] == year]
    if year_df.empty:
        print(f"No data available for the year {year}. Predicting future values...")
        
        future_year = electricity_generation_data['year'].max()
        year_df = electricity_generation_data[electricity_generation_data['year'] == future_year]
    year_df['total_electricity_production'] = year_df.groupby('country')['electricity_generation'].transform('sum')
    country_electricity_production = year_df.groupby('country')['total_electricity_production'].sum()
    top_countries = country_electricity_production.nlargest(5)
    
    return top_countries

def visualize_electricity_generation_map(input_year):

    df = pd.read_csv('dataset.csv')
    filtered_df = df[(df['year'] == input_year) & (df['electricity_generation'] >= 0) & (df['electricity_generation'] <= 3000)]
    world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
    world = world.merge(filtered_df, how='left', left_on='name', right_on='country')
    fig, ax = plt.subplots(1, 1, figsize=(15, 10))
    world.plot(column='electricity_generation', cmap='viridis', linewidth=0.8, ax=ax, edgecolor='0.8', legend=True)
    plt.title(f'Electricity Generation by Country for the year {input_year}')
    plt.axis('off')

    return fig

def get_input_year():
    input_year = simpledialog.askinteger("Input", "Enter the year:")
    if input_year:
        fig = visualize_electricity_generation_map(input_year)
        display_graph_in_window(fig)

def display_graph_in_window(fig):
    graph_window = tk.Toplevel()
    graph_window.title("Electricity Generation Map")
    canvas = FigureCanvasTkAgg(fig, master=graph_window)
    canvas.draw()
    canvas.get_tk_widget().pack(side=tk.TOP, fill=tk.BOTH, expand=1)

def analyze_energy_sources(country, year):

    country_year_df = df[(df['country'] == country) & (df['year'] == year)]

    if country_year_df.empty:
        print(f"No data available for {country} in {year}")
        return
    
    total_consumption = country_year_df['primary_energy_consumption'].values[0]
    total_energy_sources = ['coal_consumption', 'gas_consumption', 'oil_consumption', 'nuclear_consumption', 'hydro_consumption', 'solar_consumption', 'wind_consumption', 'biofuel_consumption']
    total_energy_share = [country_year_df[source].values[0] / total_consumption * 100 for source in total_energy_sources]

    fig, ax = plt.subplots(figsize=(10, 6))
    wedges, texts = ax.pie(total_energy_share, startangle=90, colors=plt.cm.tab20.colors, textprops={'fontsize': 8})
    ax.set_title(f'Share of Energy Sources in Total Energy Consumption in {country} ({year})')

    centre_circle = plt.Circle((0, 0), 0.70, fc='white')
    fig.gca().add_artist(centre_circle)

    legend_labels = [f"{source}: {share:.1f}%" for source, share in zip(total_energy_sources, total_energy_share)]
    
    ax.legend(wedges, legend_labels, loc="center", fontsize=8)

    ax.axis('equal')
    plt.tight_layout()
    return fig

def visualize_energy_sources_in_gui():
    
    country = simpledialog.askstring("Country", "Enter the country:")
    year = simpledialog.askinteger("Year", "Enter the year:")
    
    fig = analyze_energy_sources(country, year)
    
    graph_window = tk.Toplevel()
    graph_window.title(f'Energy Sources in {country} ({year})')
    
    canvas = FigureCanvasTkAgg(fig, master=graph_window)
    canvas.draw()
    canvas.get_tk_widget().pack()
def visualize_renewables_consumption():
    df = pd.read_csv('dataset.csv')

    year = int(entry_year.get())
    filtered_data = df[df['year'] == year]

    X = filtered_data[['country', 'renewables_consumption']]

    y = filtered_data['country']

    label_encoder = LabelEncoder()
    X.loc[:, 'country'] = label_encoder.fit_transform(X['country'])

    clf = GaussianNB()
    clf.fit(X, y)

    consumption_dict = {}
    for country in [entry_country1.get(), entry_country2.get(), entry_country3.get(), entry_country4.get()]:
        consumption_dict[country] = X[X['country'] == label_encoder.transform([country])[0]]['renewables_consumption'].sum()

    max_consumption_country = max(consumption_dict, key=consumption_dict.get)

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.pie(consumption_dict.values(), labels=consumption_dict.keys(), autopct='%1.1f%%', startangle=140)
    ax.set_title('Renewables Consumption Distribution in {}\nCountry with the Highest Consumption: {}'.format(year, max_consumption_country))
    ax.axis('equal')

    plot_window = Toplevel(root)
    plot_window.title("Renewables Consumption Visualization")

    # Add the plot to the new window
    canvas = FigureCanvasTkAgg(fig, master=plot_window)
    canvas.draw()
    canvas.get_tk_widget().pack()

def Rc_ip():

    input_window = Toplevel(root)
    input_window.title("Renewables Consumption Visualization Inputs")

    Label(input_window, text="Country 1:").grid(row=0, column=0, padx=5, pady=5)
    global entry_country1
    entry_country1 = Entry(input_window)
    entry_country1.grid(row=0, column=1, padx=5, pady=5)

    Label(input_window, text="Country 2:").grid(row=1, column=0, padx=5, pady=5)
    global entry_country2
    entry_country2 = Entry(input_window)
    entry_country2.grid(row=1, column=1, padx=5, pady=5)

    Label(input_window, text="Country 3:").grid(row=2, column=0, padx=5, pady=5)
    global entry_country3
    entry_country3 = Entry(input_window)
    entry_country3.grid(row=2, column=1, padx=5, pady=5)

    Label(input_window, text="Country 4:").grid(row=3, column=0, padx=5, pady=5)
    global entry_country4
    entry_country4 = Entry(input_window)
    entry_country4.grid(row=3, column=1, padx=5, pady=5)

    Label(input_window, text="Year:").grid(row=4, column=0, padx=5, pady=5)
    global entry_year
    entry_year = Entry(input_window)
    entry_year.grid(row=4, column=1, padx=5, pady=5)

    Button(input_window, text="Visualize", command=visualize_renewables_consumption, bg="#4CAF50", fg="white", padx=20, pady=10).grid(row=5, column=0, columnspan=2, pady=10)

def cluster_and_visualize(year, features):
    data = df[df['year'] == year]

    selected_features = data[features]
    
    scaler = StandardScaler()
    normalized_features = scaler.fit_transform(selected_features)

    kmeans = KMeans(n_clusters=3, random_state=42)
    data['cluster'] = kmeans.fit_predict(normalized_features)
    
    data['cluster'] = kmeans.labels_
    
    cluster_window = tk.Toplevel()
    cluster_window.title(f'Clustering Visualization for Year {year}')
  
    fig_2d = plt.figure(figsize=(6, 6))
    ax_2d = fig_2d.add_subplot(111)
    sns.scatterplot(x=features[0], y=features[1], hue='cluster', data=data, palette='viridis', ax=ax_2d)
    ax_2d.set_xlabel(features[0])
    ax_2d.set_ylabel(features[1])
    ax_2d.set_title(f'2D Clustering Visualization for Year {year}')
    canvas_2d = FigureCanvasTkAgg(fig_2d, master=cluster_window)
    canvas_2d.draw()
    canvas_2d.get_tk_widget().pack(side=tk.LEFT)
    
    fig_3d = plt.figure(figsize=(6, 6))
    ax_3d = fig_3d.add_subplot(111, projection='3d')
    scatter = ax_3d.scatter(data[features[0]], data[features[1]], data[features[2]], c=data['cluster'], cmap='viridis')
    ax_3d.set_xlabel(features[0])
    ax_3d.set_ylabel(features[1])
    ax_3d.set_zlabel(features[2])
    legend1 = ax_3d.legend(*scatter.legend_elements(), title="Cluster")
    ax_3d.add_artist(legend1)
    ax_3d.set_title(f'3D Clustering Visualization for Year {year}')
    canvas_3d = FigureCanvasTkAgg(fig_3d, master=cluster_window)
    canvas_3d.draw()
    canvas_3d.get_tk_widget().pack(side=tk.RIGHT)

def cluster_button_callback():
    year_input = simpledialog.askinteger("Year Input", "Enter the year:")
    if year_input:
        features = ['solar_consumption', 'wind_consumption', 'hydro_consumption']
        cluster_and_visualize(year_input, features)
    else:
        messagebox.showwarning("Warning", "Please enter a valid year.")


root = tk.Tk()
root.title("Energy Data Analysis")
root.geometry("800x600")

from tkinter import messagebox

try:
    bg_image = Image.open("electric-background-ajls9e7ox1fxljz2.jpg")
    bg_image = bg_image.resize((1300, 1000), Image.LANCZOS)
    background_image = ImageTk.PhotoImage(bg_image)
    background_label = tk.Label(root, image=background_image)
    background_label.place(relwidth=1, relheight=1)
    
except FileNotFoundError:
    messagebox.showerror("Error", "Background image not found.")
except Exception as e:
    messagebox.showerror("Error", f"An error occurred: {str(e)}")

title_label = tk.Label(root, text="Electricity Data Analysis", font=("Harlow Solid Itali", 28))
title_label.pack(pady=20)

button1 = tk.Button(root, text="Generation vs Demand", command=open_input_window, bg="#008CBA", fg="black", font=("Harlow Solid Itali", 10), width=20, height=2)
button1.pack(pady=10)

button2 = tk.Button(root, text="Top 5 Countries by Electricity Production", command=open_input_window_top_countries, bg="#f44336", fg="black",font=("Harlow Solid Itali", 10), width=30, height=2)
button2.pack(pady=10)

button3 = tk.Button(root, text="Visualize Electricity Generation Map", command=get_input_year, bg="#FFD700", fg="black",font=("Harlow Solid Itali", 10), width=30, height=2)
button3.pack(pady=10)

predict_button = tk.Button(root, text="Predict Electricity Generation", command=predict_electricity_generation, bg="#4CAF50", fg="black",font=("Harlow Solid Itali", 10), width=30, height=2)
predict_button.pack(pady=10)

analyze_button = tk.Button(root, text="Analyze Energy Sources", command=visualize_energy_sources_in_gui, bg="#FFA500", fg="black",font=("Harlow Solid Itali", 10), width=30, height=2)
analyze_button.pack(pady=10)

class_but = tk.Button(root, text="Analyse Renewables Consumption", command=Rc_ip, bg="#800080", fg="black",font=("Harlow Solid Itali", 10), width=30, height=2)
class_but.pack(pady=10)

ab = tk.Button(root, text="Cluster", command=cluster_button_callback, bg="#FFB6C1", fg="black", font=("Harlow Solid Itali", 10),width=30, height=2)
ab.pack(pady=(10, 30))

root.mainloop()
